# BTC Reproducible Multimodal Pipeline
Walk-forward CV + purged untouched final test. Start with validation, then baselines, then deep models.

## Deep model v2
This version uses a true VAE reconstruction objective, sinusoidal positional encoding before Transformer attention, and contiguous 48-hour validation/test sequences that include purge rows only as feature context. The final deep model is selected by walk-forward CV and evaluated once on the untouched holdout.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!pip install -q pandas numpy pyarrow scikit-learn joblib pyyaml torch

In [12]:
%cd "/content/drive/MyDrive/BTC reproducible multimodal pipeline"
!python run_pipeline.py --config config/btc_1h.yaml --stage validate

/content/drive/MyDrive/BTC reproducible multimodal pipeline

Dataset ready
  rows: 8759
  date: 2025-09-05 00:00:00+00:00 -> 2026-09-04 22:00:00+00:00
  market features: 27
  sentiment features: 10
  dev: 7444 | final purge gap: 1 | final test: 1314
  walk-forward folds: 5


In [13]:
import json, pandas as pd
out = "/content/drive/MyDrive/BTC reproducible multimodal pipeline/outputs/BTC_1h_target1h_sentiment_v4_ablation_200ep"
print(json.load(open(f"{out}/validation_report.json")))
print(json.load(open(f"{out}/split_manifest.json")))
df = pd.read_parquet(f"{out}/aligned_btc_market_sentiment.parquet")
print(df.shape, df.timestamp.min(), "->", df.timestamp.max())
display(df.head())

{'rows_before_dropna': 8783, 'first_timestamp': '2025-09-04 00:00:00+00:00', 'last_timestamp': '2026-09-04 22:00:00+00:00', 'duplicate_timestamps': 0, 'infinite_values': 0, 'missing_by_column': {'sentiment_score_lag_1h': 1, 'sentiment_score_lag_6h': 6, 'sentiment_score_lag_24h': 24}, 'target_distribution': {'1': 4395, '0': 4388}}
{'development_start': '2025-09-05 00:00:00+00:00', 'development_end': '2026-07-12 03:00:00+00:00', 'final_test_start': '2026-07-12 05:00:00+00:00', 'final_test_end': '2026-09-04 22:00:00+00:00', 'walk_forward_gap_hours': 1, 'final_test_gap_hours': 1, 'final_test_gap_start': '2026-07-12 04:00:00+00:00', 'final_test_gap_end': '2026-07-12 04:00:00+00:00', 'final_test_gap_rows': 1, 'folds': [{'fold': 1, 'train_start': '2025-09-05 00:00:00+00:00', 'train_end': '2025-10-26 18:00:00+00:00', 'validation_start': '2025-10-26 20:00:00+00:00', 'validation_end': '2025-12-17 11:00:00+00:00', 'train_rows': 1243, 'validation_rows': 1240}, {'fold': 2, 'train_start': '2025-09-0

,timestamp,close_timestamp,asset,symbol,open,high,low,close,volume,quote_volume,...,sentiment_score,sentiment_label,positive_share,negative_share,neutral_share,has_news,coverage_available,sentiment_score_lag_1h,sentiment_score_lag_6h,sentiment_score_lag_24h
0,2025-09-05 00:00:00+00:00,2025-09-05 00:59:59.999000+00:00,BTC,BTCUSDT,110730.87,110929.30,110435.75,110451.62,457.22173,5.059984e+07,...,0.000000,None,0.0,0.0,0.0,0,1,-0.676787,-0.050987,0.000000
1,2025-09-05 01:00:00+00:00,2025-09-05 01:59:59.999000+00:00,BTC,BTCUSDT,110451.63,111136.00,110451.63,111121.51,336.74632,3.731146e+07,...,0.000000,None,0.0,0.0,0.0,0,1,0.000000,-0.377987,0.605158
2,2025-09-05 02:00:00+00:00,2025-09-05 02:59:59.999000+00:00,BTC,BTCUSDT,111121.51,111482.09,111040.79,111340.65,606.85268,6.754967e+07,...,0.743411,positive,1.0,0.0,0.0,1,1,0.000000,0.663947,0.476203
3,2025-09-05 03:00:00+00:00,2025-09-05 03:59:59.999000+00:00,BTC,BTCUSDT,111340.64,111340.65,111096.67,111311.77,264.75507,2.944260e+07,...,-0.385124,negative,0.0,1.0,0.0,1,1,0.743411,-0.145867,0.322667
4,2025-09-05 04:00:00+00:00,2025-09-05 04:59:59.999000+00:00,BTC,BTCUSDT,111311.78,111468.20,111144.51,111445.00,785.66095,8.749268e+07,...,0.000000,None,0.0,0.0,0.0,0,1,-0.385124,0.000000,-0.001300


## Baselines
Run after validation looks correct.

In [14]:
!python run_horizon_suite.py --stage baselines --horizons 1 6 24


Dataset ready
  rows: 8759
  date: 2025-09-05 00:00:00+00:00 -> 2026-09-04 22:00:00+00:00
  market features: 27
  sentiment features: 10
  dev: 7444 | final purge gap: 1 | final test: 1314
  walk-forward folds: 5

Best baseline by CV: hist_gradient_boosting market_only
Final holdout metrics: {'accuracy': 0.5228310502283106, 'precision': 0.5413223140495868, 'recall': 0.39280359820089955, 'f1': 0.45525629887054736, 'pr_auc': 0.540306640321264, 'mcc': 0.05149297652453508, 'roc_auc': 0.5293906369844444}


In [15]:
!python run_horizon_suite.py --stage deep --horizons 1 6 24


Dataset ready
  rows: 8759
  date: 2025-09-05 00:00:00+00:00 -> 2026-09-04 22:00:00+00:00
  market features: 27
  sentiment features: 10
  dev: 7444 | final purge gap: 1 | final test: 1314
  walk-forward folds: 5

Best baseline by CV: hist_gradient_boosting market_only
Final holdout metrics: {'accuracy': 0.5228310502283106, 'precision': 0.5413223140495868, 'recall': 0.39280359820089955, 'f1': 0.45525629887054736, 'pr_auc': 0.540306640321264, 'mcc': 0.05149297652453508, 'roc_auc': 0.5293906369844444}
/content/drive/MyDrive/BTC reproducible multimodal pipeline/src/pipeline.py:499: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.3198738  -0.423429   -0.2846517  ... -0.29023338 -0.50793851
 -0.5217816 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tr.loc[:, cols] = scaler.transform(tr[cols])
/content/drive/MyDrive/BTC reproducible multimodal pipeline/src/pipeline.py:500: 

In [16]:
import pandas as pd
r = pd.read_csv(f"{out}/all_baseline_cv_results.csv")
display(r.groupby(["model","feature_set"])[["accuracy","f1","roc_auc","pr_auc","mcc"]].agg(["mean","std"]))

accuracy                  f1  \
                                                  mean       std      mean   
model                  feature_set                                           
hist_gradient_boosting market_only            0.517903  0.010443  0.561136   
                       market_plus_sentiment  0.509677  0.012244  0.556432   
logistic_regression    market_only            0.517742  0.011305  0.557263   
                       market_plus_sentiment  0.504032  0.011175  0.537253   
random_forest          market_only            0.510161  0.018146  0.541447   
                       market_plus_sentiment  0.500806  0.016713  0.535966   

                                                         roc_auc            \
                                                   std      mean       std   
model                  feature_set                                           
hist_gradient_boosting market_only            0.066018  0.528905  0.021324   
                       market_plus_sentiment  0.072050  0.524788  0.020942   
logistic_regression    market_only            0.055278  0.527552  0.018759   
                       market_plus_sentiment  0.074642  0.521014  0.021206   
random_forest          market_only            0.105910  0.528395  0.016939   
                       market_plus_sentiment  0.106098  0.523573  0.018266   

                                                pr_auc                 mcc  \
                                                  mean       std      mean   
model                  feature_set                                           
hist_gradient_boosting market_only            0.526532  0.027271  0.041875   
                       market_plus_sentiment  0.525062  0.029253  0.026963   
logistic_regression    market_only            0.523750  0.025084  0.036750   
                       market_plus_sentiment  0.519968  0.024873  0.010046   
random_forest          market_only            0.523730  0.023733  0.029260   
                       market_plus_sentiment  0.520017  0.024000  0.004563   

                                                        
                                                   std  
model                  feature_set                      
hist_gradient_boosting market_only            0.019392  
                       market_plus_sentiment  0.025752  
logistic_regression    market_only            0.022721  
                       market_plus_sentiment  0.023059  
random_forest          market_only            0.037324  
                       market_plus_sentiment  0.035597

## Full deep-learning pipeline
This runs LSTM, GRU, and VAE+Transformer across the walk-forward folds.

In [17]:
!python run_pipeline.py --config config/btc_1h.yaml --stage all


Dataset ready
  rows: 8759
  date: 2025-09-05 00:00:00+00:00 -> 2026-09-04 22:00:00+00:00
  market features: 27
  sentiment features: 10
  dev: 7444 | final purge gap: 1 | final test: 1314
  walk-forward folds: 5

Best baseline by CV: hist_gradient_boosting market_only
Final holdout metrics: {'accuracy': 0.5228310502283106, 'precision': 0.5413223140495868, 'recall': 0.39280359820089955, 'f1': 0.45525629887054736, 'pr_auc': 0.540306640321264, 'mcc': 0.05149297652453508, 'roc_auc': 0.5293906369844444}
/content/drive/MyDrive/BTC reproducible multimodal pipeline/src/pipeline.py:499: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.3198738  -0.423429   -0.2846517  ... -0.29023338 -0.50793851
 -0.5217816 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  tr.loc[:, cols] = scaler.transform(tr[cols])
/content/drive/MyDrive/BTC reproducible multimodal pipeline/src/pipeline.py:500: 

## V4 sentiment-only ablation

Each hourly horizon now compares **market only**, **sentiment only**, and **market + sentiment**. Deep models use max **200 epochs** with validation-loss early stopping (patience 20).

In [ ]:
!python run_horizon_suite.py --stage validate --horizons 1 6 24

In [ ]:
!python run_horizon_suite.py --stage deep --horizons 1 6 24

## Daily paper-style sentiment experiment

This compares next-day **direction**, **return**, and **close-price** prediction using market-only, sentiment-only, and combined inputs. It also includes naive persistence/zero-return baselines.

In [ ]:
!python run_daily_paper_style.py --config config/btc_daily_paper_style.yaml --stage validate

In [ ]:
!python run_daily_paper_style.py --config config/btc_daily_paper_style.yaml --stage all